# 出租车轨迹数据：拥堵 / 接客状态处理 Pipeline

来源：`E:\summercamp\出租车数据` 项目里 2026-07-23~25 那一轮讨论沉淀下来的代码，整理成 Colab / Jupyter 可跑的 notebook。

对应原来讨论里的 **Part 1-5**：

1. 起讫点（OD）判定算法设计——三态状态机
2. 拥堵CSV转矢量点（这里给的是开源版 geopandas 实现；原始 arcpy 版只能在装了 ArcGIS Pro 的机器上跑，Colab 用不了，见 Part 2 末尾说明）
3. 研究区筛选——按研究区面挑出"合格车辆"
4. 打包成端到端流水线（对应 `od_pipeline_package/run_all.py`）
5. 起终点拆分 / 完整轨迹线导出

**怎么用这个 notebook：**
- 直接跑一遍会用**合成数据**（脚本自动生成的假GPS点）演示整条流水线，几秒钟跑完，方便你确认逻辑对不对
- 想用真实数据时，把 Part 6 里的路径改成你自己的 CSV / 研究区面文件路径即可（本地 Jupyter 直接读 `E:\summercamp\...`；Colab 的话需要先把数据传到 Drive 或直接上传）


## 环境安装

Colab 默认没有 `geopandas`/`shapely`/`pyogrio`，本地 Jupyter 如果还没装也运行这一格。已经装过的话可以跳过。


In [1]:
# Colab / 本地 Jupyter 通用：装依赖（已装过可跳过，重复跑不会报错）
%pip install -q pandas numpy geopandas shapely pyogrio


Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import LineString, Point

pd.set_option("display.max_columns", None)


---
## Part 1 — 起讫点（OD）判定算法

字段假设：`id`（车辆）、`timestamp`、`lon`/`lat`、`congestion`（0/1 是否拥堵）、`pickup`（是否载客，`occupied_value` 之外的值都算"空车"）。

**判定规则**（三态状态机，逐车辆按时间排序后扫描相邻两行）：
- 起点：`pickup` 发生 `0→1` 跳变，且跳变前那一行（空车状态下）`congestion==1`（拥堵）——起点必须"空车 + 拥堵"
- 终点：从合法起点开始，往后第一次出现 `1→0` 跳变（不要求终点这一行拥堵）
- 一辆车可以有多段行程；起点条件不满足的 `0→1` 跳变不会产生虚假终点；到数据末尾还没等到终点的行程单独记到 `incomplete_trips`，不丢弃

这是原脚本 `出租车数据\od_congestion_extraction\extract_od_trips.py` 的核心逻辑，这里去掉了 argparse（notebook 里不需要命令行参数），其余算法原样保留。


In [3]:
EARTH_RADIUS_KM = 6371.0088

# 合理性阈值：单看"隐含速度"不够——如果时间戳被错记（真实数据里出现过 +6年 的脏数据），
# 时长被拉得极长会把"隐含速度"算得很小，反而看起来"合理"。所以速度和时长两个上限都要卡。
MAX_PLAUSIBLE_KMH = 150.0
MAX_PLAUSIBLE_DURATION_SEC = 86400  # 1 天


def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def find_trips_for_vehicle(occ, congestion):
    """occ: bool 数组(True=载客), congestion: 对齐的拥堵标记数组。
    返回 (trips, incomplete)：trips 是该车（已按时间排序的）切片里的
    (start_idx, end_idx) 行号对列表；incomplete 是没等到终点的起点行号列表。
    """
    n = len(occ)
    trips, incomplete = [], []
    if n < 2:
        return trips, incomplete

    change_idx = np.where(occ[1:] != occ[:-1])[0] + 1
    run_starts = np.concatenate(([0], change_idx))
    run_ends = np.concatenate((change_idx, [n]))  # exclusive

    for rs, re in zip(run_starts, run_ends):
        if not occ[rs]:
            continue  # 空车段，跳过
        if rs == 0:
            continue  # 没有前一行可验证拥堵前提，跳过
        if congestion[rs - 1] != 1:
            continue  # 起点不拥堵，不合法
        if re < n:
            trips.append((rs, re))  # re 是跳变后第一行空车（终点）
        else:
            incomplete.append(rs)  # 数据在乘客未下车前截断
    return trips, incomplete


def extract_od_trips(df, id_col="id", time_col="timestamp", lon_col="lon", lat_col="lat",
                      congestion_col="congestion", pickup_col="pickup", occupied_value="1",
                      already_sorted=False, want_trajectory=False):
    """对整份逐点表跑起讫点提取，返回 (trips_df, incomplete_df, trajectory_df|None)。"""
    required = [id_col, time_col, lon_col, lat_col, congestion_col, pickup_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"缺少必要列: {missing}，实际列: {list(df.columns)}")

    if not already_sorted:
        df = df.sort_values([id_col, time_col], kind="mergesort")
    df = df.reset_index(drop=True)

    occ_all = df[pickup_col].astype(str) == str(occupied_value)
    congestion_all = pd.to_numeric(df[congestion_col], errors="coerce")

    trip_rows, incomplete_rows, trajectory_rows = [], [], []
    trip_id = 0

    for vid, idx in df.groupby(id_col, sort=False).indices.items():
        idx = np.sort(np.asarray(idx))
        occ = occ_all.to_numpy()[idx]
        congestion = congestion_all.to_numpy()[idx]
        trips, incomplete = find_trips_for_vehicle(occ, congestion)
        sub = df.iloc[idx]

        for start_i, end_i in trips:
            trip_id += 1
            srow, erow = sub.iloc[start_i], sub.iloc[end_i]
            duration_sec = float(erow[time_col]) - float(srow[time_col])
            dist_km = haversine_km(srow[lon_col], srow[lat_col], erow[lon_col], erow[lat_col])
            implied_kmh = (dist_km / (duration_sec / 3600.0)) if duration_sec > 0 else float("inf")
            is_plausible = bool(0 < duration_sec <= MAX_PLAUSIBLE_DURATION_SEC and implied_kmh <= MAX_PLAUSIBLE_KMH)
            trip_rows.append({
                "trip_id": trip_id, id_col: vid,
                "start_time": srow[time_col], "start_lon": srow[lon_col], "start_lat": srow[lat_col],
                "end_time": erow[time_col], "end_lon": erow[lon_col], "end_lat": erow[lat_col],
                "duration_sec": duration_sec, "straight_line_km": dist_km, "is_plausible": is_plausible,
            })
            if want_trajectory:
                span = sub.iloc[start_i:end_i + 1]
                for seq, (_, prow) in enumerate(span.iterrows()):
                    trajectory_rows.append({
                        "trip_id": trip_id, id_col: vid, "seq": seq,
                        "is_endpoint": seq == 0 or seq == len(span) - 1,
                        "time": prow[time_col], "lon": prow[lon_col], "lat": prow[lat_col],
                    })

        for start_i in incomplete:
            srow = sub.iloc[start_i]
            incomplete_rows.append({id_col: vid, "start_time": srow[time_col],
                                     "start_lon": srow[lon_col], "start_lat": srow[lat_col]})

    trips_df = pd.DataFrame(trip_rows)
    incomplete_df = pd.DataFrame(incomplete_rows)
    trajectory_df = pd.DataFrame(trajectory_rows) if want_trajectory else None
    return trips_df, incomplete_df, trajectory_df


用合成数据验证边界情况：多段行程、不合法起点不产生虚假终点、末尾未完成行程。

In [4]:
# 合成一辆车 A 的数据：t2->t4 合法行程；t5不拥堵导致t6的0->1跳变不合法（t7的1->0不应被误配对）；
# t9 到结尾是未完成行程。外加一辆车 B：数据一开始就是"载客"状态（没有前置空车行可验证拥堵条件）。
demo_rows = [
    # id, timestamp,        lon,     lat,   congestion, pickup
    ("A", 1000, 116.40, 39.90, 1, 0),  # t1 空车+拥堵
    ("A", 1060, 116.41, 39.90, 0, 1),  # t2 起点（合法）
    ("A", 1120, 116.42, 39.91, 0, 1),
    ("A", 1180, 116.43, 39.91, 0, 0),  # t4 终点
    ("A", 1240, 116.44, 39.92, 0, 0),  # t5 空车不拥堵
    ("A", 1300, 116.45, 39.92, 0, 1),  # t6 起点（不合法，前一行不拥堵）
    ("A", 1360, 116.46, 39.93, 0, 0),  # t7 不应被当成终点配对
    ("A", 1420, 116.47, 39.93, 1, 0),  # t8 空车+拥堵
    ("A", 1480, 116.48, 39.94, 0, 1),  # t9 起点（合法，未完成）
    ("A", 1540, 116.49, 39.94, 0, 1),
    ("B", 2000, 116.50, 39.90, 0, 1),  # 一开始就是载客，无法验证起点，跳过
    ("B", 2060, 116.51, 39.90, 0, 0),
    ("B", 2120, 116.52, 39.91, 1, 0),
    ("B", 2180, 116.53, 39.91, 0, 1),  # 唯一有效起点
    ("B", 2240, 116.54, 39.92, 0, 0),  # 终点
]
demo_df = pd.DataFrame(demo_rows, columns=["id", "timestamp", "lon", "lat", "congestion", "pickup"])

trips_df, incomplete_df, _ = extract_od_trips(demo_df, occupied_value="1")
print("完整行程:")
display(trips_df)
print("未完成行程:")
display(incomplete_df)

assert len(trips_df) == 2 and len(incomplete_df) == 1, "边界情况没对上，检查逻辑"
print("边界情况全部符合预期。")


完整行程:


,trip_id,id,start_time,start_lon,start_lat,end_time,end_lon,end_lat,duration_sec,straight_line_km,is_plausible
0,1,A,1060,116.41,39.90,1180,116.43,39.91,120.0,2.036366,True
1,2,B,2180,116.53,39.91,2240,116.54,39.92,60.0,1.401360,True


未完成行程:


,id,start_time,start_lon,start_lat
0,A,1480,116.48,39.94


边界情况全部符合预期。


---
## Part 2 — CSV 转矢量点

**两个版本，选一个跑：**

- **开源版（本 notebook 用这个）**：pandas 分块读 + `geopandas.points_from_xy` 向量化建点 + `pyogrio` 批量追加写 GeoPackage。不依赖 ArcGIS，Colab 能跑。对应原来的 `od_pipeline_package/01_build_points.py`。
- **arcpy 版（仅供参考，Colab/普通Python跑不了）**：用 `arcpy.da.NumPyArrayToFeatureClass` 整块写 + `Append` 追加，比逐行 `InsertCursor` 快一个数量级；只能用 ArcGIS Pro 自带的 `python.exe`（`D:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe`）运行，原始文件在 `出租车数据\是否堵车\build_congestion_points.py`，逻辑跟下面开源版完全对应，只是写入API换成了arcpy的。

`bad_coord==1` 的行（占位0,0坐标）不生成点，只统计跳过数量。


In [5]:
def csv_chunks_to_points(csv_path_or_df, out_path=None, layer_name="pts", chunk_size=2_000_000,
                          lon_col="lon", lat_col="lat", bad_coord_col=None, crs="EPSG:4326"):
    """分块把逐点CSV(或已经在内存里的DataFrame)转成点GeoDataFrame，可选顺带写盘(GPKG)。
    大文件传路径+chunk_size做流式；小份/演示直接传DataFrame一次性转换即可。
    """
    def _to_points(df):
        if bad_coord_col and bad_coord_col in df.columns:
            df = df[df[bad_coord_col] == 0]
        geometry = gpd.points_from_xy(df[lon_col], df[lat_col])
        return gpd.GeoDataFrame(df, geometry=geometry, crs=crs)

    if isinstance(csv_path_or_df, pd.DataFrame):
        gdf = _to_points(csv_path_or_df)
        if out_path:
            gdf.to_file(out_path, layer=layer_name, driver="GPKG")
        return gdf

    total_rows, written, first = 0, 0, True
    reader = pd.read_csv(csv_path_or_df, chunksize=chunk_size)
    parts = []
    for df in reader:
        total_rows += len(df)
        gdf = _to_points(df)
        written += len(gdf)
        if out_path:
            gdf.to_file(out_path, layer=layer_name, driver="GPKG", append=not first)
            first = False
        else:
            parts.append(gdf)
        print(f"累计读取={total_rows:,} 累计写入={written:,}")
    return pd.concat(parts, ignore_index=True) if parts else None


In [6]:
# 演示：把 demo_df 转成点，congestion 作为普通属性字段(不是Z值)
demo_points = csv_chunks_to_points(demo_df, lon_col="lon", lat_col="lat")
display(demo_points.head())
print(f"hasZ per geometry: {demo_points.geometry.iloc[0].has_z}  (应为 False —— congestion 是普通属性列，不进geometry)")


,id,timestamp,lon,lat,congestion,pickup,geometry
0,A,1000,116.40,39.90,1,0,POINT (116.4 39.9)
1,A,1060,116.41,39.90,0,1,POINT (116.41 39.9)
2,A,1120,116.42,39.91,0,1,POINT (116.42 39.91)
3,A,1180,116.43,39.91,0,0,POINT (116.43 39.91)
4,A,1240,116.44,39.92,0,0,POINT (116.44 39.92)


hasZ per geometry: False  (应为 False —— congestion 是普通属性列，不进geometry)


---
## Part 3 — 研究区筛选

研究区面（`insect-area.shp`）本身是手绘/多来源相交出来的（KDE热点研究区 ∩ 拥堵道路研究区），这一步在 ArcGIS Pro 里操作，没有太多可复用代码。

这里复用的是下游那一步的代码：**按研究区面挑出"合格车辆"**——同一辆车当天只要有任意一个"载客中(`occupied`)"的点落在研究区面内，就保留该车**当天全部**GPS点（不是只留落在区内那一段），因为后面 Part 1 的起讫点状态机需要完整的前后文才能正确判断起终点。

对应原始文件 `od_pipeline_package/02_filter_by_study_area.py`，用 `shapely.contains_xy` 向量化做点在面内判断（不是逐点构造Point对象再判断，快很多）。


In [7]:
def load_study_area_geometry(path_or_geom):
    """传文件路径(.shp/.gpkg等)会自动读取并统一到WGS84；也可以直接传shapely geometry。"""
    if hasattr(path_or_geom, "geom_type"):  # 已经是 shapely geometry
        geom = path_or_geom
    else:
        gdf = gpd.read_file(path_or_geom)
        if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs("EPSG:4326")
        geom = gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union
    shapely.prepare(geom)
    return geom


def find_qualified_vehicle_ids(df, geom, id_col="id", lon_col="lon", lat_col="lat",
                                pickup_col="pickup", occupied_value=1):
    inside = shapely.contains_xy(geom, df[lon_col].to_numpy(), df[lat_col].to_numpy())
    hit = inside & (df[pickup_col].to_numpy() == occupied_value)
    return set(df.loc[hit, id_col].unique().tolist())


def filter_by_study_area(df, geom, id_col="id", lon_col="lon", lat_col="lat",
                          pickup_col="pickup", occupied_value=1, bad_coord_col=None):
    """返回：合格车辆当天全部GPS点(不只是落在区内的那一段)。"""
    qualified = find_qualified_vehicle_ids(df, geom, id_col, lon_col, lat_col, pickup_col, occupied_value)
    keep = df[df[id_col].isin(qualified)]
    if bad_coord_col and bad_coord_col in keep.columns:
        keep = keep[keep[bad_coord_col] == 0]
    return keep, qualified


In [8]:
# 演示：拿 demo_df 造一个小研究区面(方框)，只框住车辆A的坐标范围，车辆B应该被整体丢弃
from shapely.geometry import box
demo_study_area = box(116.395, 39.895, 116.445, 39.925)  # 覆盖A的起终点范围，不覆盖B

filtered_df, qualified_ids = filter_by_study_area(
    demo_df, demo_study_area, id_col="id", lon_col="lon", lat_col="lat",
    pickup_col="pickup", occupied_value=1,
)
print("合格车辆:", qualified_ids)
display(filtered_df)
assert qualified_ids == {"A"}, "应只有A落在研究区内并载客"
print("研究区筛选逻辑符合预期：车辆A全天数据保留，车辆B整体丢弃。")


合格车辆: {'A'}


,id,timestamp,lon,lat,congestion,pickup
0,A,1000,116.40,39.90,1,0
1,A,1060,116.41,39.90,0,1
2,A,1120,116.42,39.91,0,1
3,A,1180,116.43,39.91,0,0
4,A,1240,116.44,39.92,0,0
5,A,1300,116.45,39.92,0,1
6,A,1360,116.46,39.93,0,0
7,A,1420,116.47,39.93,1,0
8,A,1480,116.48,39.94,0,1
9,A,1540,116.49,39.94,0,1


研究区筛选逻辑符合预期：车辆A全天数据保留，车辆B整体丢弃。


---
## Part 4 — 打包成端到端流水线

对应 `od_pipeline_package/run_all.py`：串起来就是 **转点(Part 2, 可选) → 研究区筛车辆(Part 3) → 起讫点提取(Part 1)**，逐天跑，产出 `<day>_trips.csv`（喂给最优路径算法的最终结果）。

原始版本是分成 3 个独立 `.py` 脚本 + `subprocess` 调用（方便队友分步调试、支持断点续跑），notebook 里为了方便演示直接改成函数直连调用，逻辑完全一致。


In [9]:
def run_pipeline_for_day(df, study_area_geom, *, id_col="id", time_col="timestamp",
                          lon_col="lon", lat_col="lat", congestion_col="congestion",
                          pickup_col="pickup", occupied_value="1", bad_coord_col=None,
                          want_trajectory=False):
    """跑完 Part 3(研究区筛选) -> Part 1(起讫点提取)，返回 (trips_df, incomplete_df, filtered_df, trajectory_df)。
    Part 2(转点)是独立可选步骤，只是为了留一份点矢量文件方便可视化，不影响这里的结果，
    需要的话单独调用上面的 csv_chunks_to_points。
    """
    filtered_df, qualified_ids = filter_by_study_area(
        df, study_area_geom, id_col=id_col, lon_col=lon_col, lat_col=lat_col,
        pickup_col=pickup_col, occupied_value=int(occupied_value), bad_coord_col=bad_coord_col,
    )
    print(f"研究区筛选: {df[id_col].nunique()}辆车 -> 合格{len(qualified_ids)}辆, 保留{len(filtered_df)}行")

    trips_df, incomplete_df, trajectory_df = extract_od_trips(
        filtered_df, id_col=id_col, time_col=time_col, lon_col=lon_col, lat_col=lat_col,
        congestion_col=congestion_col, pickup_col=pickup_col, occupied_value=occupied_value,
        want_trajectory=want_trajectory,
    )
    n_implausible = int((~trips_df["is_plausible"]).sum()) if len(trips_df) else 0
    print(f"起讫点提取: 完整行程{len(trips_df)}条(其中{n_implausible}条is_plausible=False), "
          f"未完成行程{len(incomplete_df)}条")
    return trips_df, incomplete_df, filtered_df, trajectory_df


In [10]:
# 演示：对 demo_df 跑完整流水线
trips_df, incomplete_df, filtered_df, _ = run_pipeline_for_day(
    demo_df, demo_study_area, occupied_value="1", want_trajectory=False,
)
display(trips_df)


研究区筛选: 2辆车 -> 合格1辆, 保留10行
起讫点提取: 完整行程1条(其中0条is_plausible=False), 未完成行程1条


,trip_id,id,start_time,start_lon,start_lat,end_time,end_lon,end_lat,duration_sec,straight_line_km,is_plausible
0,1,A,1060,116.41,39.9,1180,116.43,39.91,120.0,2.036366,True


### 多天批量跑（对应 `run_all.py --days ...`）

真实用的时候，把 `days_data` 换成 `{日期: 对应DataFrame}`（本地读CSV：`pd.read_csv(f"{base_dir}/{day}_data/{day}_data.csv")`）即可，其余不用改。


In [11]:
def run_pipeline_all_days(days_data: dict, study_area_geom, **kwargs):
    """days_data: {日期字符串: 当天逐点DataFrame}。返回 {日期: (trips_df, incomplete_df, filtered_df, trajectory_df)}。"""
    results = {}
    for day, df in days_data.items():
        print(f"\n===== {day} =====")
        results[day] = run_pipeline_for_day(df, study_area_geom, **kwargs)
    return results

# 演示：假装有两天数据(这里偷懒两天都用同一份demo_df)
demo_results = run_pipeline_all_days({"20170301": demo_df, "20170302": demo_df}, demo_study_area, occupied_value="1")



===== 20170301 =====
研究区筛选: 2辆车 -> 合格1辆, 保留10行
起讫点提取: 完整行程1条(其中0条is_plausible=False), 未完成行程1条

===== 20170302 =====
研究区筛选: 2辆车 -> 合格1辆, 保留10行
起讫点提取: 完整行程1条(其中0条is_plausible=False), 未完成行程1条


---
## Part 5 — 起终点拆分 / 完整轨迹线导出

两个产出：

1. **起终点拆成同一份"长表"**：`trips.csv` 是宽表（一行一趟，`start_*`/`end_*` 两组列），用 `point_type` 字段（`start`/`end`）拆成长表，方便GIS里同时显示、用 `trip_id` 关联。对应 `tools/build_od_points.py`。
2. **中间轨迹点 + 连好的行程折线**：状态机跑的时候其实"看过"了起点到终点之间的每一个GPS点，只是原来没保留——加一个开关就能顺手吐出来。对应 `extract_od_trips.py` 里的 `--trip-points-output`/`--trip-lines-output`（Part 1 的 `extract_od_trips` 函数已经内置了 `want_trajectory` 参数，就是这个功能）。


In [12]:
def split_od_points(trips_df, id_col="id"):
    """宽表(start_*/end_*) -> 长表(point_type=start/end)，起终点各占一半，trip_id 关联。"""
    start_cols = [c for c in trips_df.columns if c.startswith("start_")]
    end_cols = [c for c in trips_df.columns if c.startswith("end_")]
    common_cols = [c for c in trips_df.columns if c not in start_cols and c not in end_cols]

    starts = trips_df[common_cols].copy()
    starts["point_type"] = "start"
    for c in start_cols:
        starts[c[len("start_"):]] = trips_df[c]

    ends = trips_df[common_cols].copy()
    ends["point_type"] = "end"
    for c in end_cols:
        ends[c[len("end_"):]] = trips_df[c]

    long_df = pd.concat([starts, ends], ignore_index=True)
    if "trip_id" in long_df.columns:
        long_df = long_df.sort_values(["trip_id", "point_type"])
    return long_df


def trips_to_od_points_gdf(trips_df, crs="EPSG:4326"):
    long_df = split_od_points(trips_df)
    return gpd.GeoDataFrame(long_df, geometry=gpd.points_from_xy(long_df["lon"], long_df["lat"]), crs=crs)


def trajectory_to_lines_gdf(trajectory_df, trips_df, id_col="id", crs="EPSG:4326"):
    """把 extract_od_trips(..., want_trajectory=True) 吐出来的逐点轨迹表，按trip_id连成LineString。"""
    lines = []
    for tid, group in trajectory_df.sort_values(["trip_id", "seq"]).groupby("trip_id", sort=False):
        coords = list(zip(group["lon"], group["lat"]))
        if len(coords) < 2:
            continue
        lines.append({"trip_id": tid, "n_points": len(coords), "geometry": LineString(coords)})
    lines_df = pd.DataFrame(lines).merge(
        trips_df[["trip_id", id_col, "start_time", "end_time", "duration_sec", "straight_line_km", "is_plausible"]],
        on="trip_id", how="left",
    )
    return gpd.GeoDataFrame(lines_df, geometry="geometry", crs=crs)


In [13]:
# 演示 1：起终点拆分成长表
od_points_gdf = trips_to_od_points_gdf(trips_df)
display(od_points_gdf)
assert len(od_points_gdf) == 2 * len(trips_df)
print(f"{len(trips_df)}趟行程 -> {len(od_points_gdf)}个点(起点{len(trips_df)} + 终点{len(trips_df)})")


,trip_id,id,duration_sec,straight_line_km,is_plausible,point_type,time,lon,lat,geometry
1,1,A,120.0,2.036366,True,end,1180,116.43,39.91,POINT (116.43 39.91)
0,1,A,120.0,2.036366,True,start,1060,116.41,39.90,POINT (116.41 39.9)


1趟行程 -> 2个点(起点1 + 终点1)


In [14]:
# 演示 2：重新跑一遍流水线但打开 want_trajectory=True，拿到中间轨迹点，再连成行程折线
trips_df2, incomplete_df2, filtered_df2, trajectory_df = run_pipeline_for_day(
    demo_df, demo_study_area, occupied_value="1", want_trajectory=True,
)
display(trajectory_df)

trip_lines_gdf = trajectory_to_lines_gdf(trajectory_df, trips_df2)
display(trip_lines_gdf)
print("每条行程连成一条 LineString，几何列可以直接 .to_file() 存成 .gpkg/.shp。")


研究区筛选: 2辆车 -> 合格1辆, 保留10行
起讫点提取: 完整行程1条(其中0条is_plausible=False), 未完成行程1条


,trip_id,id,seq,is_endpoint,time,lon,lat
0,1,A,0,True,1060,116.41,39.90
1,1,A,1,False,1120,116.42,39.91
2,1,A,2,True,1180,116.43,39.91


,trip_id,n_points,geometry,id,start_time,end_time,duration_sec,straight_line_km,is_plausible
0,1,3,"LINESTRING (116.41 39.9, 116.42 39.91, 116.43 ...",A,1060,1180,120.0,2.036366,True


每条行程连成一条 LineString，几何列可以直接 .to_file() 存成 .gpkg/.shp。


---
## Part 6 — 接真实数据（本地 Jupyter）

上面全程用的是合成的 `demo_df`。要跑真实的7天数据，把下面这一格的路径改成你自己的即可，其它函数不用改：

```python
base_dir = r"E:\summercamp\出租车数据\是否堵车"
days = [f"201703{d:02d}" for d in range(1, 8)]
study_area_path = r"E:\summercamp\出租车数据\study_area\newest-area\insect-area.shp"

study_area_geom = load_study_area_geometry(study_area_path)

days_data = {
    day: pd.read_csv(f"{base_dir}\\{day}_data\\{day}_data.csv")
    for day in days
}  # 注意：真实文件是 3000万行/天，本地内存要够；Colab 免费版内存有限，建议只挑1天测试

real_results = run_pipeline_all_days(
    days_data, study_area_geom,
    id_col="taxi_id", time_col="timestamp", lon_col="longitude", lat_col="latitude",
    congestion_col="congestion", pickup_col="occupied", occupied_value="1",
    bad_coord_col="bad_coord", want_trajectory=True,
)
```

**如果在 Colab 跑真实数据**：本地文件传不过去，需要先上传到 Colab 或挂载 Google Drive：

```python
from google.colab import drive
drive.mount('/content/drive')
# 然后把 base_dir / study_area_path 换成 /content/drive/... 下的路径
```

大文件（3000万行/天）建议用 `csv_chunks_to_points` 里那种分块读取模式，不要一次性 `pd.read_csv` 整个读进内存；Colab 免费版内存通常只有12GB左右，7天全跑可能不够，建议先用 `--days` 只挑1-2天验证逻辑，跑通了再考虑要不要迁回本地跑全量（本地机器实测7天全量转点约30分钟/天，起讫点提取约3分钟/天）。
